# Akbank GenAI Bootcamp: RAG Chatbot Projesi

* [Proje Amacı](#scrollTo=SV1WON2F3Mpp)
* [Detaylı RAG Test Sonuçları](#scrollTo=6kkyHs8TRfKl)
* [Gerekli Kütüphanelerin Kurulumu ve Ortam Ayarı](#scrollTo=O_NTJT3W5PP6)
* [Veri Seti Hazırlama ve Tanıtımı](#scrollTo=nJNoTblh5h3d)
* [RAG Mimarisi ve Pipeline Geliştirme](#scrollTo=YunqduwI5oFB)
* [Pipeline Kurulum Adımlarının Özeti](#scrollTo=O_JEsgAcI82r)
* [Web Arayüzü ve Ürün Kılavuzu](#scrollTo=GpR6pXHj5tL9)

## Projenin Amacı

Bu proje, RAG (Retrieval Augmented Generation) mimarisi kullanarak **Metin/WikiRAG-TR** veri seti üzerinde çalışan bir **Soru-Cevap (Q&A)** chatbot geliştirmeyi amaçlamaktadır. Proje, **Gemini API**'ı kullanarak veriye dayalı, doğru ve bağlamsal olarak zenginleştirilmiş yanıtlar sunmayı hedeflemektedir.

**Kullanılan Veri Seti:**
* https://huggingface.co/datasets/Metin/WikiRAG-TR


**Kullanılan Yöntemler ve Teknolojiler:**

* Generation Model: Gemini API
    * Google'ın geliştirdiği model olup, Gemini API ve **google-genai SDK** ile hızlıca uygulanabilmesinin, Türkçe RAG görevlerinde kullanım kolaylığı sağlamaktadır. Kullanımı için API anahtarı gerekmektedir.

* RAG Framework: LangChain
    * RAG projeleri için en yaygın kullanılan ve en geniş topluluk desteğine sahip framework'tür. Gemini API'a özgü entegrasyonları hızlıca uygulanabilir.

* Embedding Model: trmteb/turkish-embedding-model
    * Gemini API ile sorunsuz entegrasyon sağlar ve genellikle LangChain ile uyumludur. Hugging Face platformunda yer alan bu model için API anahtarının bulunması gerekmektedir.

* Vektör Veritabanı: Chroma
    * Kurulumu ve kullanımı son derece kolay, hafif (in-memory) bir veritabanıdır. Colab ortamında hızlı prototipleme ve test için idealdir.

* Web Arayüzü: Gradio
    * Python kodu üzerinden minimal çabayla profesyonel ve etkileşimli web uygulamaları oluşturmanızı sağlayan, projenin arayüz katmanı, Makine Öğrenimi demoları için optimize edilmiş ve Colab Notebook ortamında son derece stabil çalışan kütüphanedir.



## Detaylı RAG Test Sonuçları

**Sonuçlar Özeti**:

* Veriseti, 3-5 Saniyede Yüklendi.
* Toplam Öğe Sayısı 5000.
* Parçalanmış belge sayısı: 20324.
* Vektör aşamasında 2GB sistem, 1.8GB GPU RAM ihtiyaç duymuştur.
* Pipeline, 300-314 Saniyede Yüklendi.
* Zincir, 4 Saniyede Yüklendi:

### Hata Analizi

Bu testler, $k=3$ (en iyi 3 kaynak) ayarıyla yapılan sorguların sistem üzerindeki etkisini incelemektedir. Çıktılar, **Retrieval** aşamasının başarılı olduğunu, ancak bazı sorgularda **Embedding Kalitesinin** ve **Retrieval-Generation uyumunun** sorunlu olduğunu ortaya koymaktadır.

### RAG Sistemi Test Sonuçları ve Kritik Hata Analizi

Bu testler, $k=3$ ayarıyla yapılan sorguların Retrieval ve Generation aşamalarındaki performansını inceler.

#### 1. Tarih Testi (Kritik Retrieval Hatası)

* **Soru:** 1923'te Türkiye'de hangi önemli olay gerçekleşmiştir?
* **Yanıt:** Elimdeki bilgilere göre bu soruya net bir yanıt veremiyorum.
* **Çekilen Kaynak Sayısı:** 3
* **Analiz:**
    * **Hata:** Çekilen 3 kaynak da (Örn: Hitler'in denizaltı planları, Montrö) tamamen 1923 ile alakasız (1930'lar/II. Dünya Savaşı dönemi) içeriktedir.
    * **Sonuç:** Türkçe Embedding modeli, **"1923"** terimini hatalı bir şekilde genel "Türkiye Tarihi" bağlamlarıyla eşleştirmiştir. Kaynaklar alakasız olduğu için, modelin yanıt vermeyi reddetmesi (hallucination yapmaması) doğrudur.

#### 2. Dil/Etimoloji Testi (Başarılı Sonuç)

* **Soru:** Tengri sözcüğünün anlamı nedir?
* **Yanıt:** Tengri, bugünkü Türkçedeki Tanrı sözcüğünün eski söyleniş şeklidir.
* **Çekilen Kaynak Sayısı:** 3
* **Analiz:**
    * **Başarı:** Kaynak 1, 2 ve 3'ün tümü, sorunun cevabını içeren net bir bilgiyle başlamaktadır.
    * **Sonuç:** Bu örnek, bilgi setindeki net tanımlara dayalı sorgularda sistemin doğru çalıştığını kanıtlar. Retrieval ve Generation zinciri başarılıdır.

#### 3. Biyografi Testi (Kritik Embedding Hatası)

* **Soru:** Alan Tuning kimdir?
* **Yanıt:** Elimdeki bilgilere göre bu soruya net bir yanıt veremiyorum.
* **Çekilen Kaynak Sayısı:** 3
* **Analiz:**
    * **Hata:** Tüm çekilen kaynaklar, sorudaki **"Alan Tuning"** yerine, fonetik olarak benzer olan **"TuneIn"** adlı radyo uygulaması hakkındadır.
    * **Sonuç:** Embedding modeli, "Tuning" ve "TuneIn" kelimelerini fonetik benzerlik nedeniyle anlamsal olarak yanlış sınıflandırmıştır. Modelin yanlış bağlama rağmen yanıt üretmeyerek "hallucination"dan kaçınması olumlu bir davranıştır.

### Sonuç ve İyileştirme Önerileri

1.  **En Büyük Sorun: Embedding Kalitesi (Hata Tipi: Alakasız Çekme)**
    * Sistem, tarihsel yıllar ("1923") ve benzer kelimeler ("Tuning" vs. "TuneIn") konusunda güçlü anlamsal ayrımlar yapamıyor.
    * **Öneri:** Veri setinin ve embedding modelinin (trmteb) bu tür nüansları ayırt edip edemediği incelenmeli veya $k=3$ yerine $k=1$ gibi daha kısıtlı bir ayarla **en iyi tek kaynaktan** faydalanılmalıdır.
2.  **Model Davranışı (Güvenilirlik):** Gemini'nin tüm alakasız durumlarda (`1923` ve `Alan Tuning`) yanıt vermeyi reddetmesi, **güvenilirliğin yüksek olduğunu** ve zorla yanlış bilgi (hallucination) üretmediğini göstermektedir.
3.  **Başarılı Retrieval:** Tengri örneği, sistemin basit ve net tanımlarda mükemmel çalıştığını kanıtlar.



## 1. Gerekli Kütüphanelerin Kurulumu ve Ortam Ayarı

Bu adımda, projenin çalışması için gereken tüm Python kütüphaneleri (LangChain, Gradio, ChromaDB, Google GenAI SDK'sı dahil) kurulacaktır. Ayrıca, Google Gemini API anahtarı sisteme tanımlanarak API erişimi sağlanacaktır.


In [8]:
# Gerekli Kütüphanelerin Kurulumu
# -q: sessiz kurulum
# --upgrade: eğer kuruluysa yükselt
# langchain-community: ChromaDB ve diğer topluluk entegrasyonları için zorunludur.

!pip install -q --upgrade \
    langchain-huggingface \
    sentence-transformers \
    google-genai \
    langchain \
    langchain-google-genai \
    langchain-community \
    chromadb \
    datasets \
    gradio \
    requests==2.32.4 \
    pyarrow==19.0.0 \
    google-generativeai \
    google-ai-generativelanguage
print("Tüm kütüphaneler başarıyla kuruldu veya güncellendi.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Tüm kütüphaneler başarıyla kuruldu veya güncellendi.


In [9]:
import os
from google import genai
from google.genai.errors import APIError
from google.colab import userdata

# 1. API Anahtarını Yükleme (Colab Secrets Kullanımı)
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Anahtarlar başarıyla yüklendi.")
except userdata.SecretNotFoundError as e:
    print(f"Lütfen Colab Secrets menüsünde anahtarınızı tanımlayın. Hata:{e}")
    # Eğer localde çalışıyorsanız, os.environ["GEMINI_API_KEY"] = "Sizin-Anahtarınız" kullanabilirsiniz.
    # Eğer localde çalışıyorsanız, os.environ["HF_TOKEN"] = "Sizin-Anahtarınız" kullanabilirsiniz.




Anahtarlar başarıyla yüklendi.


## 2. Veri Seti Hazırlama ve Tanıtımı
Kullanılan veri seti Hugging Face üzerindeki **Metin/WikiRAG-TR**'dır. Bu set, Türkçe Vikipedi maddelerini içermektedir. Veri seti, Soru-Cevap mimarisinin temelini oluşturacak bilgiyi sağlamaktadır.



### Veri Seti Hakkında Bilgi

**İçerik:** WikiRAG-TR, Türkçe Vikipedi maddelerinin toplamda bulunan 14379 maddeden sentetik olarak oluşturulmuş 6 bin (5999) soru-cevap çiftinden oluşan bir veri kümesidir. Veri kümesi, Türkçe Arama-Artırılmış Üretim (RAG) görevlerinde kullanılmak üzere oluşturulmuştur.


**Bilgiler:**
* Veri setindeki 5999 verinin içindeki 5725 adet sentetik soru-cevap çifti, 274 artırılmış negatif örnek veri şeklinde oluşturulmuş.
* Seri setinin boyutu 20.5 MB şeklindedir ve bilgiler türkçedir.

**Proje Konusu ile İlişkisi:** Geliştireceğimiz RAG temelli chatbot, bu veri setini "bilgi kaynağı" (Knowledge Base) olarak kullanacaktır. Kullanıcının sorduğu sorulara yanıt aranırken, sadece bu Vikipedi tabanlı Türkçe metin parçaları içinde arama yapılacaktır.

---

> **NOT**: Üretilen yanıtlar genellikle kısa ve özdür. Bu, bu veri kümesi üzerinde eğitilen modellerin kısa yanıtlar üretmesine yol açabilir. Bu veri kümesini oluşturmak için Wikipedia makaleleri kullanıldığından, bu veri kümesinde bulunan önyargılar ve yanlışlıklar bu veri kümesinde de mevcut olabilir.





#### Hazırlanış Metodolojisi

Veri setinin oluşturulması iki ana aşamada gerçekleştirilmiş ve her aşama ayrı bir diyagramla gösterilmiştir.




##### **Alt Kategori Koleksiyonu**
!["collecting_subcategories"](https://huggingface.co/datasets/Metin/WikiRAG-TR/resolve/main/docs/collecting_subcategories.png)

Bu aşamada veriler, içeriklerine göre alt sınıflar içerecek şekilde sınıflandırılmıştır. Maddeler kısaca aşağıda belirtilmiştir:

* Fen, teknoloji, mühendislik, matematik, fizik, kimya, biyoloji, jeoloji, meteoroloji, tarih, sosyal bilimler ve daha fazlasını içeren, özenle seçilmiş bir temel kategori listesi oluşturulmuştur.
* Bu temel kategoriler kullanılarak, alt kategoriler Vikipedi'den yinelemeli olarak toplanmıştır.
* Yineleme derinliği 3 olarak ayarlandı ve her derinlik katmanı için toplanacak alt kategori sayısı 100 ile sınırlandırılmıştır.
* Her adımda, aşağıdaki alt kategori türleri filtrelenmiştir:
  * Müstehcen kelimeler içeren alt kategoriler.
  * Yalnızca öğe listeleri içeren alt kategoriler.
  * Şablon olarak kullanılan alt kategoriler.
  * Ortaya çıkan alt kategori listesinden makaleler alınmıştır.




##### **Veri Seti Oluşturma**

!["creating_wikirag_tr"](https://huggingface.co/datasets/Metin/WikiRAG-TR/resolve/main/docs/creating_wikirag_tr.png)

Bu aşamada aşağıdaki maddeler ile oluşturulmuştur:

* Giriş bölümleri, 1. Aşamada toplanan makalelerden çıkarılmıştır.
  * Giriş bölümü çok kısa veya çok uzunsa (50'den az veya 2500'den fazla karakter), makale atılmıştır.
  * Giriş bölümü uygunsuz (NSFW) kelimeler içeriyorsa, makale atılmıştır.
  * Giriş bölümü denklemler içeriyorsa, makale atılmıştır.
  * Giriş bölümü boşsa, makale atılmıştır.
* Filtrelenen girişler, sentetik soru-cevap çiftleri oluşturmak için büyük bir dil modeline (Gemma-2-27B-it) aktarılmıştır.
* Veri kümesinde (giriş, soru ve cevap içeren) ortaya çıkan her satır için aşağıdaki işlemler gerçekleştirilmiştir:
  * Bağlama yanlış pozitif sonuçlar eklemek için diğer satırlardan ilgisiz bağlamlar (girişler) toplandı.
  * Bu ilgisiz bağlamlar bir listeye eklendi.
  * İlgili bağlam bu listeye eklendi. (Bazı durumlarda, yanıtın modelin yetersiz bilgi nedeniyle soruyu yanıtlayamayacağını gösterdiği negatif örnekler oluşturmak için ilgili bağlam çıkarılmıştır. Bu negatif örnekler ayrı ayrı oluşturularak, tüm orijinal soruların karşılık gelen yanıtlara sahip olması sağlanmıştır.)
  * Liste, ilgili bağlamın konumunu rastgele belirlemek için karıştırılmıştır.
  * Liste öğeleri '\n' karakteri kullanılarak birleştirilmiştir.

#### Veri Kümesi Sütunları

Proje kapsamında kullanılacak veri seti **Metin/WikiRAG-TR**'dır. Bu veri seti, sadece kaynak metni değil, aynı zamanda Soru-Cevap çiftlerini ve zengin meta verileri de içermektedir.

### Veri Seti Sütun Yapısı:

| Sütun Adı | Açıklama | RAG'deki Kullanımı |
| :--- | :--- | :--- |
| **context** | Hem ilgili hem de ilgisiz bilgileri içeren genişletilmiş bağlam. | **Temel RAG Kaynağı (Page Content)** |
| **id** | Her satır için benzersiz tanımlayıcı. | **Metadata** |
| **question** | Model tarafından oluşturulan soru. | Eğitim sonrası analiz/testlerde kullanılabilir. |
| **answer** | Model tarafından oluşturulan cevap. | Eğitim sonrası analiz/testlerde kullanılabilir. |
| **is_negative_response** | Cevabın olumsuz bir cevap olup olmadığını belirtir (0: Hayır, 1: Evet). | **Metadata** |
| Diğer Sütunlar | Bağlamın parçalanmasına ve analizine yardımcı olan teknik bilgiler. | **Metadata** |


Bizim RAG sürecimizde, **context** sütunundaki metinleri alıp LangChain **Document** objelerine dönüştüreceğiz ve diğer önemli sütunları da **metadata** olarak saklayacağız.

### Veri Seti Yükleme ve Önizleme

Aşağıdaki kod bloğu, Hugging Face datasets kütüphanesi aracılığıyla **Metin/WikiRAG-TR** veri setinin yüklenmesini ve yapısının incelenmesini sağlar.

In [10]:
from datasets import load_dataset
from langchain.docstore.document import Document
from typing import List

def load_dataset_and_preview(dataset_name: str, select: int = 5000) -> List[Document]:
    """
    Belirtilen Hugging Face veri setini yükler, ilk 'select' kadarını alır ve
    LangChain Document objeleri listesine dönüştürür.
    İstenen sütunlar metadata olarak saklanır.

    Args:
        dataset_name (str): Yüklenecek Hugging Face veri setinin adı (Örn: "Metin/WikiRAG-TR").
        select (int): Yüklenecek belge sayısı. Prototip için varsayılan 5000'dir.

    Returns:
        List[Document]: LangChain formatına dönüştürülmüş belgelerin listesi.
    """

    # Hugging Face'den veri setini yükleme
    print("Veri seti yükleniyor...")
    # split="train" ile tren setini yüklüyor ve ilk 'select' kadarını alıyoruz.
    dataset = load_dataset(dataset_name, split="train",token=os.getenv("HF_TOKEN")).select(range(select))
    print("Veri seti başarıyla yüklendi.")

    # Toplam Öğe Sayısı
    N_DOCUMENTS = len(dataset)
    print(f"Toplam Öğe Sayısı: {N_DOCUMENTS}")

    all_documents = []

    # Metadata olarak saklanacak sütunlar
    METADATA_COLS = ['id', 'question', 'answer']

    for item in dataset:
        # Kaynak metin için 'context' sütununu kullanıyoruz
        page_content = item.get('context', 'Context Not Found') # Güvenlik için get() kullandık

        # Metadata objesini oluşturma
        metadata = {col: item.get(col) for col in METADATA_COLS}
        metadata['source'] = dataset_name # Veri seti adını dinamik olarak kaynağa yazıyoruz

        # LangChain Document objesini oluşturma
        doc = Document(page_content=page_content, metadata=metadata)
        all_documents.append(doc)

    print(f"\nLangChain formatına dönüştürülen toplam belge sayısı: {len(all_documents)}")

    # İlk 3 Metin Parçasının LangChain Document Formatında Önizlemesi
    print("\n--- İlk 3 Metin Parçasının LangChain Document Formatında Önizlemesi ---")
    for doc in all_documents[:3]:
        print(f"ID: {doc.metadata.get('id', 'Yok')}")
        print(f"Question (Metadata): {doc.metadata.get('question', 'Yok')}")
        print(f"Answer (Metadata): {doc.metadata.get('answer', 'Yok')[:100]}...")
        print(f"İçerik (Context - İlk 100 Karakter): {doc.page_content[:100]}...")
        print("-" * 30)

    return all_documents

## 3. RAG Mimarisi ve Pipeline Geliştirme (LangChain, Gemini ve Chroma)
Bu adımda, RAG mimarimizi LangChain'i birleştirici framework olarak kullanarak inşa edeceğiz. Mimari temel olarak üç ana bileşenden oluşur:

1.  **Embedding ve Indexing (Vektörleştirme ve Dizinleme):** **WikiRAG-TR** veri setindeki **context** metinleri, **trmteb/turkish-embedding-model** Modeli kullanılarak sayısal vektörlere dönüştürülür ve hızlı arama için **Chroma Vektör Veritabanı**'na kaydedilir.
2.  **Retrieval (Geri Çağırma):** Kullanıcı bir soru sorduğunda, bu soru vektörleştirilir ve ChromaDB'de en alakalı (anlamsal olarak en yakın) metin parçaları geri çağrılır.
3.  **Generation (Üretim):** Geri çağrılan metinler (context) ve orijinal kullanıcı sorusu, etkili bir prompt şablonu (Prompt Template) içine yerleştirilir ve **Gemini API**'a gönderilir. Gemini, sadece bu bağlama dayanarak yanıtı üretir.



### Embedding Modelinin Tanımlanması ve Vektör Veritabanı Oluşturma (Indexing)

Bu aşama, RAG sisteminin bellek (Memory) bileşenini oluşturur. Daha önceki denemelerde karşılaşılan `503 Illegal metadata` ve zaman aşımı sorunlarını aşmak ve Türkçe performansı optimize etmek amacıyla, Google API'sine bağımlı kalınmamıştır.

**Yapılan Başlıca Optimizasyonlar:**

1.  **Yerel Embedding Modelinin Kullanımı:** Gemini API'si yerine, Türkçe için özel olarak eğitilmiş **`trmteb/turkish-embedding-model`** (HuggingFace) modeli benimsenmiştir. Bu model, LangChain'in `HuggingFaceEmbeddings` sınıfı aracılığıyla yüklenmiş ve Colab ortamında hızlı bir şekilde oluşturmak için **CUDA'ya zorlanmıştır (`device='cuda'`)**.
2.  **Parçalama (Chunking):** Toplam 5000 orijinal doküman, **`RecursiveCharacterTextSplitter`** kullanılarak 1000 karakterlik küçük parçalara (chunks) ayrılmıştır. Bu, hem LLM'ye giden yükü azaltmış hem de `Illegal metadata` hatalarını çözmüştür.
3.  **Vektör Veritabanı:** Parçalanmış tüm dokümanlar, ChromaDB'ye kaydedilmiştir. Bu yoğun indexleme süreci, sistemin gelecekteki bilgi çekme (retrieval) operasyonlarının temelini oluşturmuştur.

Bu işlemlerin sonucunda, tüm Türkçe bilgi seti vektörleştirilmiş ve RAG zincirinin sorgulara hızlıca cevap verebileceği bir formata dönüştürülmüştür.


In [11]:
from typing import List, Tuple

# LangChain sınıflarının içe aktarılması
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter # KRİTİK DEĞİŞİM
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema.retriever import BaseRetriever # Retriever'ın ana sınıfı



def setup_rag_pipeline(
    all_documents: List[Document],
    model_name: str = "trmteb/turkish-embedding-model"
) -> Tuple[HuggingFaceEmbeddings, Chroma, BaseRetriever]:
    """
    RAG pipeline'ı için embedding modelini tanımlar, belgeleri vektörleştirir
    ve ChromaDB'ye kaydeder, ardından bir retriever döndürür.

    Döndürülen değerler (Tuple):
    - GoogleGenAIEmbeddings: Kullanılan embedding modelinin nesnesi.
    - Chroma: Vektör veritabanının nesnesi.
    - BaseRetriever: Arama yapabilen retriever nesnesi.
    """
    print("Vektör veritabanı ve RAG zinciri kuruluyor...")

    # BELGE PARÇALAMA (CHUNKING)
    print("Belgeler parçalanıyor (Chunking)...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        # Ayırıcı listesi belirtilmediğinde varsayılan akıllı listeyi kullanır
    )

    # Parçalanmış belgeleri oluşturma
    chunked_documents = text_splitter.split_documents(all_documents)
    print(f"Orijinal belge sayısı: {len(all_documents)}. Parçalanmış belge sayısı: {len(chunked_documents)}")

    # 1. Google Embedding Modelini Tanımlama
    print(f"Yerel Türkçe Embedding modeli yükleniyor: {model_name}")
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True}
    )

    # 2. Vektör Veritabanı Oluşturma (Indexing)
    print(f"Toplam {len(all_documents)} belge vektörleştirilip ChromaDB'ye kaydediliyor. Lütfen bekleyiniz...")

    vectorstore = Chroma.from_documents(
        documents=chunked_documents,
        embedding=embeddings,
        collection_name="wikirag-tr-rag"
    )

    print("\nVektörleştirme ve Indexing (Dizinleme) işlemi tamamlandı!")
    print(f"Chroma Veritabanı (Collection: 'wikirag-tr-rag') hazırlandı.")

    # 3. Retriever (Geri Çağırıcı) Tanımlama
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}
    )

    print("Retriever (Geri Çağırıcı) LangChain için tanımlandı.")

    # İstenen 3 nesneyi Tuple olarak döndürüyoruz
    return embeddings, vectorstore, retriever

### Generation Modelini, Prompt'u ve RAG Chain'i Tanımlama

Bu aşama, RAG sisteminin akıl yürütme (Reasoning) ve sohbet yönetimi (Chat Management) bileşenlerini birleştirir.

**Kurulum Detayları:**

1.  **Generation Modeli (LLM):** Yanıt üretimi için Google'ın hızlı olan **`gemini-2.5-flash`** modeli seçilmiştir. Bu model, `Timeout` sorunlarını aşmak amacıyla stabilite için **60 saniyelik kesin bir zaman aşımı (`timeout: 60`)** ile yapılandırılmıştır.
2.  **Retrieval Optimizasyonu:** Vektör veritabanından bilgi çekme (Retrieval) işlemi, sorgu başına en alakalı 3 belgeyi (k=3) çekecek şekilde ayarlanmıştır.$k=3$ Neden Tercih Edildi? $k=1$ (tek belge) aşırı kısıtlayıcı olabileceği ve karmaşık soruları yanıtlamak için yeterli bağlam sağlayamayabileceği için, $k=3$ seçimi, LLM'e daha geniş ve güvenilir bir bilgi yelpazesi sunarak yanıtların hem doğru hem de kapsamlı olmasını sağlamayı hedeflemiştir.
3.  **Sohbet Yetenekli Prompt:** Sisteme, sadece soru-cevap değil, aynı zamanda bağlamı koruma yeteneği kazandırmak için **`ChatPromptTemplate`** kullanılmıştır. Bu prompt, hem çekilen `context`'i hem de kullanıcının `chat_history`'sini aynı anda LLM'ye iletecek şekilde tasarlanmıştır. Bu sayede chatbot, önceki konuşmaları hatırlayabilen, akıcı bir sohbete dönüşmüştür.
4.  **RAG Zincirinin Kurulumu:** Son olarak, `create_retrieval_chain` ve `create_stuff_documents_chain` yapıları kullanılarak, Retrieval (Bilgi Çekme) ve Generation (Yanıt Üretme) adımlarını birleştiren **nihai RAG zinciri (`rag_chain`)** oluşturulmuştur. Bu zincir, gelen her sorguyu işleyip doğru kaynaklara dayalı bir yanıt üretmekten sorumludur.

Bu kurulum, projenin hem bilgiye dayalı doğruluk (Contextual Accuracy) hem de sohbet akıcılığı gereksinimlerini karşılamaktadır.


In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.schema.retriever import BaseRetriever
from langchain.schema.runnable import Runnable

def setup_model_and_chain(retriever: BaseRetriever, model_name: str = "gemini-2.5-flash") -> Runnable:
    """
    Gemini LLM, Prompt Template ve Retriever kullanarak LangChain RAG zincirini (Runnable) oluşturur.

    Args:
        retriever (BaseRetriever): ChromaDB'den gelen ve arama yapabilen Retriever nesnesi.
        model_name (str): Kullanılacak Gemini modelinin adı.

    Returns:
        Runnable: Tüm RAG akışını içeren LangChain zinciri (LCEL uyumlu ana arayüz).
    """

    # 1. LLM (Generative Model) Tanımlama
    print(f"{model_name} modeli tanımlanıyor...")
    llm = ChatGoogleGenerativeAI(
        model=model_name,
        temperature=0.0,
        request_options={"timeout": 60}
    )

    # 2. Prompt Şablonu Oluşturma
    prompt_template = """Sen bir Türkçe RAG (Bilgiye Dayalı Soru-Cevap) asistanısın.
Aşağıdaki 'context' (bağlam) içinde yer alan bilgileri kullanarak kullanıcının sorusunu yanıtla.
Eğer bağlamda soruya net bir cevap bulamıyorsan, kibarca 'Elimdeki bilgilere göre bu soruya net bir yanıt veremiyorum.' şeklinde cevap ver.
Asla verilen bağlam dışına çıkma ve kendi genel bilginle cevap verme.

Context:
---
{context}
---

Soru: {input}
Yanıt:
"""

    prompt = ChatPromptTemplate.from_template(prompt_template)

    # 3. Dokümanları Birleştirme Zinciri (Document Combination Chain) Oluşturma
    document_chain = create_stuff_documents_chain(llm, prompt)

    # 4. Ana RAG Zincirini (Chain) Oluşturma
    rag_chain = create_retrieval_chain(retriever, document_chain)

    print("RAG Pipeline (Zinciri) başarıyla oluşturuldu!")

    # 5. Pipeline Testi
    print("\n--- Pipeline Test Ediliyor ---")
    test_queries = ["1923'te Türkiye'de hangi önemli olay gerçekleşmiştir?","Tengri sözcüğünün anlamı nedir?","Alan Tuning kimdir?"]
    for test_query in test_queries:
      response = rag_chain.invoke({"input": test_query})

      print(f"\nSoru: {test_query}")
      print(f"\nGemini Yanıtı:\n{response['answer']}")
      print("\n--- Kullanılan Kaynaklar (Context) ---\n")
      for i, doc in enumerate(response['context']):
          print(f"Kaynak {i+1} (ID: {doc.metadata.get('id', 'N/A')} - Soru: {doc.metadata.get('question', 'Yok')[:40]}...):")
          # Kaynak içeriğinin ilk 500 karakterini yazdırır
          print(f"İçerik Başlangıcı: {doc.page_content.strip()[:500]}...")
          print("-" * 30)

    return rag_chain

## Pipeline Kurulum Adımlarının Özeti

Bu kod bloğu, RAG (Retrieval-Augmented Generation) sisteminin arka planını, yani LLM'ye güç veren bilgi tabanını ve tüm akış zincirini kurar.

| Aşama | Kod | Amaç |
| :--- | :--- | :--- |
| **1. Veri Yükleme** | `load_dataset_and_preview(...)` | Veriyi LangChain Document'a çevirir. |
| **2. Indexing** | `setup_rag_pipeline(...)` | Türkçe Embeddings ile ChromaDB'ye vektör kaydeder. |
| **3. Chain Kurulumu** | `setup_model_and_chain(...)` | Gemini ve Retriever'ı birleştirir. |

Bu adımların sırasıyla ve hatasız tamamlanması, chatbot'un sorulara doğru ve bağlama dayalı cevap verebilmesi için kritik öneme sahiptir.





In [13]:
from datetime import datetime
# Pipeline'ı yükle
try:
    # Veri Setini Yükleme ve Dokümanlara Dönüştürme
    start_time = datetime.now()
    print("\nVeriler Hazırlanıyor...")
    all_documents = load_dataset_and_preview(dataset_name="Metin/WikiRAG-TR",select=5000)
    dataset_load_time = (datetime.now() - start_time)
    print(f"\nVeriseti, {dataset_load_time.seconds} Saniyede Yüklendi: ")
    print("="*30)

    # 2. Embedding ve Indexing
    start_time = datetime.now()
    print("\nRAG Pipeline Hazırlanıyor...")
    embeddings,vectorstore,retriever = setup_rag_pipeline(all_documents,model_name="trmteb/turkish-embedding-model")
    pipeline_load_time = (datetime.now() - start_time)
    print(f"\nPipeline, {pipeline_load_time.seconds} Saniyede Yüklendi: ")
    print("="*30)

     # 3. LLM ve Chain Tanımlama
    start_time = datetime.now()
    print("\nModel ve Retriever Hazırlanıyor...")
    rag_chain = setup_model_and_chain(retriever,model_name="gemini-2.5-flash")
    setup_chain_time = (datetime.now() - start_time)
    print(f"\nZincir, {setup_chain_time.seconds} Saniyede Yüklendi: ")
    print("="*30)

except Exception as e:
    print(f"RAG Pipeline Kurulumunda Hata: {e}")




Veriler Hazırlanıyor...
Veri seti yükleniyor...
Veri seti başarıyla yüklendi.
Toplam Öğe Sayısı: 5000

LangChain formatına dönüştürülen toplam belge sayısı: 5000

--- İlk 3 Metin Parçasının LangChain Document Formatında Önizlemesi ---
ID: fdb9e733-8b3f-430e-93d4-72c563f2d00c
Question (Metadata): Bermuda Adaları'nın Birleşik Krallık'a bağlı bir bölge olmasının tarihi sebepleri nelerdir?
Answer (Metadata): Bermuda Adaları, 1609 yılında İngiliz denizci George Somers tarafından keşfedilmiştir. O zamandan be...
İçerik (Context - İlk 100 Karakter): Thingspiel (çoğulu Thingspiele), 1930'larda savaş öncesi Nazi Almanyası'nda kısa süreli popülerlik k...
------------------------------
ID: cb5dacdf-cb15-49c5-b0a5-8de9bb77cb48
Question (Metadata): .bm uzantısının kullanımı ne zaman başladı?
Answer (Metadata): .bm uzantısı 2007 yılında kullanıma açıldı....
İçerik (Context - İlk 100 Karakter): Kabulgan – Türk, Altay ve Moğol mitolojisinde “Şekil Değiştirme” kavramı. Metamorfoz, transformasyon...
--

## 4. Web Arayüzü ve Ürün Kılavuzu (Gradio)

Proje, **Gradio** kullanılarak minimal çabayla, kullanıcı dostu ve etkileşimli bir web arayüzü ile sunulacaktır. Bu arayüz, kullanıcıların RAG chatbot'un yeteneklerini kolayca test etmesini sağlar.



### Web Arayüzü Mimarisi

Arayüz, daha önce oluşturulan LangChain RAG pipeline'ını (`rag_chain`) arka planda çalıştırır. Kullanıcıdan gelen her sorgu, doğrudan bu pipeline'a iletilir ve Gemini API'dan gelen yanıt, ekranda gösterilir. Ayrıca, yanıtın dayandırıldığı kaynak metinler (retriever'dan gelen `context` belgeleri) de şeffaflık sağlamak amacıyla kullanıcının görebileceği şekilde sunulacaktır.

      Büyük olasılıkla `https://localhost:7862/` bağlantısı üzerinden erişilebilir hale getirecektir.

In [14]:
import gradio as gr
import os
from langchain_core.messages import HumanMessage, AIMessage
from langchain_huggingface import HuggingFaceEmbeddings

# =========================================================================
# GRADIO ARAYÜZÜNÜ BAŞLATMA
# =========================================================================


def chat_interface_handler(query: str, history: list) -> str:
    """
    Gradio'dan gelen yeni sorguyu ve sohbet geçmişini alır,
    RAG zincirini çalıştırır ve yanıtı döndürür.
    """

    # LangChain'in anlayacağı formata (HumanMessage/AIMessage) çevirme
    # Gradio history formatı: [[user_msg, bot_msg], [user_msg, bot_msg], ...]
    langchain_history = []
    for human, ai in history:
        langchain_history.append(HumanMessage(content=human))
        langchain_history.append(AIMessage(content=ai))

    # Güncel sorguyu ekleme
    langchain_history.append(HumanMessage(content=query))

    try:
        # RAG Zincirini çalıştırma
        # LangChain'in 'runnable' zincirleri, history'yi 'chat_history' anahtarıyla alabilir.
        response = rag_chain.invoke({
            "input": query,
            "chat_history": langchain_history
        })

        # Kaynakları ve yanıtı ayırma
        answer = response['answer']
        context_docs = response['context']

        # Kaynakları daha okunaklı bir formatta (HTML) yanıta ekleme
        sources_markdown = "🔍 Kaynaklar:\n"

        for i, doc in enumerate(context_docs):
            content = doc.page_content.replace('\n', ' ').strip()
            doc_id = doc.metadata.get('id', 'N/A')

            # Markdown içindeki details etiketi (Gradio'nun daha iyi işlediği format)
            sources_markdown += f"""
<details>
<summary>Kaynak {i+1} (ID: {doc_id} - Q: {doc.metadata.get('question', 'Yok')[:50]}...)</summary>
{content}
</details> """
        final_response_with_sources = f"{answer}<br><br>{sources_markdown}"

        return final_response_with_sources

    except Exception as e:
        return f"Sorgu sırasında hata oluştu: {e}"


# gr.ChatInterface bileşenini kullanma
chat_interface = gr.ChatInterface(
    fn=chat_interface_handler,
    textbox=gr.Textbox(placeholder="Sormak istediğiniz soruyu buraya yazın...", container=False, scale=7),
    title="🇹🇷 Gemini RAG Chatbot (Sohbet Modu)",
    description="Türkçe Yerel Embedding ve Gemini 2.5 Flash ile desteklenen, sohbet geçmişini takip eden RAG uygulaması.",
    theme="soft",
    examples=[
        ["Türkiye Cumhuriyeti'ni kim kurdu?"],
        ["Kurucunun siyasi hayatı hakkında bilgi ver."],
        ["Bu kişi hangi savaşlara katıldı?"]
    ]
)

# Gradio'yu başlatma
print("Gradio Sohbet Arayüzü başlatılıyor...")
chat_interface.launch(inline=True, share=False)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Gradio Sohbet Arayüzü başlatılıyor...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>